# 01 — Khám phá dữ liệu tài chính (EDA)

Notebook này phân tích sơ bộ dataset trước khi đưa vào Moirai:
- Kiểm tra chất lượng dữ liệu (missing values, outliers)
- Visualize chuỗi thời gian
- Phân tích tính stationarity và seasonality

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dotenv import load_dotenv
import os

load_dotenv('../.env')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12

## 1. Load dữ liệu

Đặt file CSV vào `data/raw/` rồi cập nhật các biến bên dưới.

In [ ]:
# ============ CẤU HÌNH — thay đổi theo file của bạn ============
CSV_FILE  = os.path.join('..', os.getenv('DATA_DIR', 'data/raw'), 'your_data.csv')
DATE_COL  = os.getenv('DATE_COL', 'Date')
TARGET_COL = os.getenv('TARGET_COL', 'Close')
FREQ      = os.getenv('FREQ', 'B')
# ================================================================

df = pd.read_csv(CSV_FILE, parse_dates=[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)
print(f'Shape: {df.shape}')
df.head()

## 2. Kiểm tra chất lượng dữ liệu

In [ ]:
print('=== Missing values ===')
print(df.isnull().sum())
print()
print('=== Thống kê mô tả ===')
df[[TARGET_COL]].describe()

In [ ]:
# Kiểm tra khoảng cách ngày (gap detection)
date_diffs = df[DATE_COL].diff().dt.days.dropna()
print(f'Khoảng cách ngày — min: {date_diffs.min()}, max: {date_diffs.max()}, median: {date_diffs.median()}')
gaps = date_diffs[date_diffs > date_diffs.median() * 3]
if len(gaps) > 0:
    print(f'\nPhát hiện {len(gaps)} gap lớn:')
    print(df.loc[gaps.index, DATE_COL].values)
else:
    print('Không có gap bất thường.')

## 3. Visualize chuỗi thời gian

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Toàn bộ chuỗi
axes[0].plot(df[DATE_COL], df[TARGET_COL], linewidth=0.8, color='steelblue')
axes[0].set_title(f'{TARGET_COL} — Toàn bộ chuỗi ({len(df)} điểm)')
axes[0].set_ylabel(TARGET_COL)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30)

# 200 điểm gần nhất (context window Moirai)
recent = df.tail(200)
axes[1].plot(recent[DATE_COL], recent[TARGET_COL], linewidth=1.2, color='darkorange')
axes[1].set_title('200 điểm gần nhất (context window Moirai)')
axes[1].set_ylabel(TARGET_COL)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('../results/eda_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Phân phối giá trị & outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df[TARGET_COL].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Phân phối giá trị')
axes[0].set_xlabel(TARGET_COL)

returns = df[TARGET_COL].pct_change().dropna() * 100
returns.hist(bins=50, ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Phân phối % thay đổi (returns)')
axes[1].set_xlabel('Return (%)')

plt.tight_layout()
plt.savefig('../results/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Volatility (std of returns): {returns.std():.2f}%')
print(f'Skewness: {returns.skew():.3f} | Kurtosis: {returns.kurtosis():.3f}')

## 5. Tóm tắt cho luận văn

In [ ]:
summary = {
    'Tổng số điểm': len(df),
    'Ngày bắt đầu': df[DATE_COL].min().strftime('%Y-%m-%d'),
    'Ngày kết thúc': df[DATE_COL].max().strftime('%Y-%m-%d'),
    'Tần suất': FREQ,
    'Missing values': df[TARGET_COL].isnull().sum(),
    'Min': f"{df[TARGET_COL].min():.2f}",
    'Max': f"{df[TARGET_COL].max():.2f}",
    'Mean': f"{df[TARGET_COL].mean():.2f}",
    'Std': f"{df[TARGET_COL].std():.2f}",
}
for k, v in summary.items():
    print(f'{k:25s}: {v}')